# Block Diagram

## Series, Parallel and Feedback

In [76]:
from sympy import symbols, prod,simplify, Eq, latex, solve, solveset

import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,
})

from mathprint import *

In [ ]:
series.__doc__ = "Series connection: multiply transfer functions in cascade."
parallels.__doc__ = "Parallel connection: sum transfer functions connected in parallel."
negative_feedbacks.__doc__ = "Negative feedback: compute G/(1 + G*H) for one or more feedback paths."
positive_feedbacks.__doc__ = "Positive feedback: compute G/(1 - G*H) for one or more feedback paths."

series, parallels, negative_feedbacks, positive_feedbacks

In [77]:
def series(*argv):
    return prod(argv)

def parallels(*argv):
    return sum(argv)

def negative_feedbacks(G, *argv):
    ret = G
    for k in range(len(argv)):
        ret = ret / (1 + ret * argv[k])

    return ret

def positive_feedbacks(G, *argv):
    ret = G
    for k in range(len(argv)):
        ret = ret / (1 - ret * argv[k])
    
    return ret

## Example 1

Simplify the following block diagram:

![](./images/diagram1.png)

Our first step is to modify the block diagram such that it contains ony three connection configurations:  

- serial connections
- parallel connections
- feedback connections (positive or negative)

The next figure presents the modified block diagram that is composed only by those three connections. 

![](./images/diagram2.png)


```
M1 = series(G3, G4)
M2 = positive_feedbacks(M1, H4)
M3 = series(M2, G2)
M4 = negative_feedbacks(M3, H3/G4)
M5 = series(M4, G1)
G = negative_feedbacks(M5, H3/G4, H2/G4, H1)
```
Or we can combine them into one line of codes: 
```
G = negative_feedbacks(series(negative_feedbacks(series(positive_feedbacks(series(G3, G4), H4), G2), H3/G4), G1), H3/G4, H2/G4, H1)
```

In [78]:
G1, G2, G3, G4, H1, H2, H3, H4 = symbols('G1 G2 G3 G4 H1 H2 H3 H4')

G = negative_feedbacks(series(negative_feedbacks(series(positive_feedbacks(series(G3, G4), H4), G2), H3/G4), G1), H3/G4, H2/G4, H1);
display(simplify(G))

G1*G2*G3*G4/(G1*G2*G3*G4*H1 + G1*G2*G3*H2 + G1*G2*G3*H3 + G2*G3*H3 - G3*G4*H4 + 1)

## Example 2

![](./images/p24.png)

This problem is very difficult to solve with graphical method. Instead, we will use general algebra to solve the problem with SymPy.

- There are at least 6 new intermediate variables: $e_1$, $e_2$, $y_1$, $y_2$, $y_3$ and $C$.
- We need to setup at least 6 equations (see the red labels).
- Solve those equations for all intermediate variables.

In [116]:
G1, G2, G3, G4   = symbols('G1 G2 G3 G4')
E, R, C = symbols('E R C')
e1, e2, y1, y2, y3 = symbols('e1 e2 y1 y2 y3')

eq1 = Eq(R-C, e1)
eq2 = Eq(e1-H1*y2, e2)
eq3 = Eq(G1*e2, y1)
eq4 = Eq(G2*(y1-H2*C), y2)
eq5 = Eq(G3*y2, y3)
eq6 = Eq(y3-G4*e2, C)


mprint(latex(eq1))
mprint(latex(eq2)) 
mprint(latex(eq3))
mprint(latex(eq4))
mprint(latex(eq5))
mprint(latex(eq6))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [115]:
ans = solve((eq1, eq2, eq3, eq4, eq5, eq6),  (C, e1, e2, y1, y2, y3))
for k in ans.keys():
    mprint(latex(k), '=', latex(ans[k]))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Now we take $C = \dots$ and divide it by $R$ to obtain $C/R$.

In [117]:
tf = simplify(ans[C]/R)
mprintb("\\frac{C}{R} =", latex(tf))

<IPython.core.display.Math object>